In [1]:
# [CELL 1] KHAI BÁO CẤU HÌNH VÀ BIẾN MÔI TRƯỜNG
import os

# --- 1. CẤU HÌNH DEEPGRAM ---
DEEPGRAM_API_KEYS = [
    "fad53b39ac51343050733aea2a36c0c2f8a37a08", # profile dev
    "e89600bec269c64cfe620b23694aed9162e20324", # profile kenkunkanki
    "33c67dee44f8dc9580e441d8b85dca2a31e628c2", # profile hanmunmun
]

# --- 2. CẤU HÌNH TẢI DỮ LIỆU TỪ GOOGLE DRIVE ---
DATA_INPUT_GDRIVE_URL = "https://drive.google.com/open?id=1fHPdgk0FeiY1kGaWj9eGHCjHICwud7cZ"

# --- 3. CẤU HÌNH RCLONE ---
RCLONE_REMOTE_NAME = "codevcn-dell-pc"
DATASET_RCLONE_CONF = "/kaggle/input/datasets/kenkunkanki/rclone-config/rclone.conf"

# --- 4. CẤU HÌNH ĐƯỜNG DẪN THƯ MỤC LÀM VIỆC ---
WORKING_DIR = "/kaggle/working"
DOWNLOAD_DIR = os.path.join(WORKING_DIR, "audio_files")
OUTPUT_DIR = os.path.join(WORKING_DIR, "output")
WORKING_RCLONE_CONF = os.path.join(WORKING_DIR, "rclone.conf")

print("✅ [Cell 1] Đã khởi tạo các biến cấu hình thành công.")

✅ [Cell 1] Đã khởi tạo các biến cấu hình thành công.


In [2]:
import requests
import time
from typing import Literal, Optional

# ==========================================
# CẤU HÌNH KẾT NỐI MÁY CHỦ
# ==========================================
REMOTE_SERVER_URL = "https://b6-remote-server-kaggle-2026.onrender.com/webhook/notebook"

# Khuyến nghị: Bạn nên lưu SERVER_API_KEY vào Kaggle Secrets để bảo mật, 
# sau đó gọi ra bằng: from kaggle_secrets import UserSecretsClient ...
SERVER_API_KEY = "b6-remote-server-kaggle-2026"

# Mã định danh cho toàn bộ chuỗi công việc (Bạn có thể tự đặt tên cho mỗi lần chạy)
JOB_ID = "broccoli-01"

def send_webhook_to_server(
    index_type: Literal["start", "mid", "end"],
    status: Literal["started", "completed"],
    progress: Optional[str] = None,
    next_notebook: Optional[str] = None
) -> bool:
    """
    Hàm gửi tín hiệu điều phối về máy chủ trung tâm trên Render.
    """
    headers = {
        "Content-Type": "application/json",
        "X-API-Key": SERVER_API_KEY
    }
    
    payload = {
        "job_id": JOB_ID,
        "notebook_index_type": index_type,
        "status": status,
        "progress": progress,
        "next_notebook_ref": next_notebook
    }
    
    print(f"\n[{time.strftime('%H:%M:%S')}] Đang gửi tín hiệu [{status.upper()}] cho vị trí [{index_type.upper()}] đến Remote Server...")
    
    try:
        # Sử dụng timeout=60 để chờ Render khởi động nếu đang ở trạng thái ngủ
        response = requests.post(REMOTE_SERVER_URL, json=payload, headers=headers, timeout=60)
        response.raise_for_status()
        print(f"[{time.strftime('%H:%M:%S')}] THÀNH CÔNG: Máy chủ phản hồi '{response.json().get('message')}'")
        return True
        
    except requests.exceptions.Timeout:
        print(f"[{time.strftime('%H:%M:%S')}] LỖI: Máy chủ phản hồi quá chậm (Vượt quá 60 giây).")
        return False
    except requests.exceptions.HTTPError as err:
        print(f"[{time.strftime('%H:%M:%S')}] LỖI HTTP: Bị từ chối truy cập hoặc sai API Key. Chi tiết: {err}")
        return False
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] LỖI HỆ THỐNG: Không thể kết nối. Chi tiết: {str(e)}")
        return False

In [3]:
send_webhook_to_server(index_type="start", status="started")


[18:38:33] Đang gửi tín hiệu [STARTED] cho vị trí [START] đến Remote Server...


[18:39:05] LỖI HỆ THỐNG: Không thể kết nối. Chi tiết: HTTPSConnectionPool(host='b6-remote-server-kaggle-2026.onrender.com', port=443): Max retries exceeded with url: /webhook/notebook (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e579d783b90>: Failed to resolve 'b6-remote-server-kaggle-2026.onrender.com' ([Errno -3] Temporary failure in name resolution)"))


False

In [ ]:
# [CELL 2] CÀI ĐẶT THƯ VIỆN VÀ PHỤ THUỘC (DEPENDENCIES)
!pip install deepgram-sdk gdown -q
!curl https://rclone.org/install.sh | sudo bash
print("✅ [Cell 2] Đã cài đặt xong các thư viện và công cụ cần thiết.")

In [ ]:
# [CELL 3] DỌN DẸP VÀ CHUẨN BỊ THƯ MỤC LÀM VIỆC
import shutil
import os

def cleanup_and_prepare_dirs():
    """Xóa bỏ các thư mục làm việc cũ (nếu có) và khởi tạo thư mục mới trống rỗng."""
    for directory in [DOWNLOAD_DIR, OUTPUT_DIR]:
        if os.path.exists(directory):
            print(f"[*] Đang dọn dẹp thư mục cũ: {directory}")
            shutil.rmtree(directory)
        
        os.makedirs(directory, exist_ok=True)
        print(f"[*] Đã khởi tạo thư mục mới: {directory}")

# Thực thi dọn dẹp
cleanup_and_prepare_dirs()
print("✅ [Cell 3] Quá trình dọn dẹp và chuẩn bị thư mục đã hoàn tất.")

In [ ]:
# [CELL 4] TẢI DỮ LIỆU TỪ GOOGLE DRIVE (GDOWN + RCLONE FALLBACK TRỰC TIẾP QUA ID)
import gdown
import subprocess
import glob
import shutil
import os
import stat
import re

def extract_gdrive_id(url):
    """Trích xuất tự động Folder ID từ đường dẫn Google Drive."""
    match = re.search(r"[/=]([-\w]{25,})", url)
    if match:
        return match.group(1)
    if re.match(r"^[-\w]{25,}$", url):
        return url
    return None

def setup_rclone_conf():
    """Copy file rclone.conf từ Dataset sang /kaggle/working/ và cấp quyền ghi."""
    if not os.path.exists(DATASET_RCLONE_CONF):
        print(f"[!] Lỗi: Không tìm thấy file gốc tại {DATASET_RCLONE_CONF}")
        return False
        
    try:
        shutil.copy2(DATASET_RCLONE_CONF, WORKING_RCLONE_CONF)
        os.chmod(WORKING_RCLONE_CONF, stat.S_IRUSR | stat.S_IWUSR)
        print(f"[*] Đã copy và cấp quyền Write cho rclone.conf tại: {WORKING_RCLONE_CONF}")
        return True
    except Exception as e:
        print(f"[!] Lỗi khi thiết lập file cấu hình rclone: {str(e)}")
        return False

def flatten_directory(target_dir, extension=".wav"):
    """Đưa toàn bộ file .wav từ các thư mục con ra ngoài thư mục gốc."""
    print("[*] Đang cấu trúc lại thư mục tải về (Flattening)...")
    for root, dirs, files in os.walk(target_dir):
        if root == target_dir:
            continue
        for file in files:
            if file.endswith(extension):
                source_path = os.path.join(root, file)
                dest_path = os.path.join(target_dir, file)
                
                if os.path.exists(dest_path):
                    base, ext = os.path.splitext(file)
                    folder_name = os.path.basename(root)
                    dest_path = os.path.join(target_dir, f"{base}_{folder_name}{ext}")
                    
                shutil.move(source_path, dest_path)
    
    for root, dirs, files in os.walk(target_dir, topdown=False):
        if root != target_dir and not os.listdir(root):
            os.rmdir(root)
    print("[*] Đã xử lý xong, tất cả file âm thanh hiện nằm tại thư mục gốc.")

def download_audio_files():
    """Tải file với gdown, nếu thất bại dùng rclone để ép tải thẳng từ thư mục mang ID đó."""
    print("[*] Phân tích đường dẫn Google Drive...")
    folder_id = extract_gdrive_id(DATA_INPUT_GDRIVE_URL)
    
    if not folder_id:
        print(f"[!] Lỗi nghiêm trọng: Không thể trích xuất ID từ đường dẫn cung cấp: {DATA_INPUT_GDRIVE_URL}")
        return False
        
    print(f"[*] Đã nhận diện được Folder ID: {folder_id}")
    print("[*] Bắt đầu tiến trình tải file qua gdown...")
    is_success = False
    
    try:
        gdown.download_folder(id=folder_id, output=DOWNLOAD_DIR, quiet=False, use_cookies=False)
        wav_files = glob.glob(f"{DOWNLOAD_DIR}/**/*.wav", recursive=True)
        
        if len(wav_files) > 0:
            is_success = True
            print(f"[*] Đã tải thành công {len(wav_files)} file qua mạng gdown.")
        else:
            print("[!] Không tìm thấy file .wav qua gdown. Hệ thống sẽ chuyển qua rclone.")
    except Exception as e:
        print(f"[!] Gặp sự cố khi sử dụng gdown: {str(e)}")
    
    # --- Rclone Fallback sử dụng trực tiếp Folder ID ---
    if not is_success:
        print("[*] Chuyển đổi phương thức sang rclone fallback...")
        if not setup_rclone_conf():
            return False
            
        try:
            # Ghi đè root_folder_id để tải đúng vào thư mục theo ID mà không cần biết cấu trúc đường dẫn
            rclone_source = f"{RCLONE_REMOTE_NAME}:"
            cmd = [
                "rclone", "copy", rclone_source, DOWNLOAD_DIR, 
                "--config", WORKING_RCLONE_CONF, 
                "--drive-root-folder-id", folder_id,
                "--stats", "5s"
            ]
            
            print(f"[*] Đang thực thi lệnh tải bằng rclone cho ID: {folder_id}...")
            subprocess.run(cmd, check=True)
            
            wav_files_fallback = glob.glob(f"{DOWNLOAD_DIR}/**/*.wav", recursive=True)
            if len(wav_files_fallback) > 0:
                print(f"[*] Đã tải dữ liệu thành công qua rclone ({len(wav_files_fallback)} file).")
            else:
                 print(f"[!] Rclone đã chạy nhưng không tìm thấy file .wav nào trong thư mục đích.")
        except subprocess.CalledProcessError as e:
            print(f"[!] Lỗi nghiêm trọng khi chạy lệnh rclone: {e}")
            return False
    
    flatten_directory(DOWNLOAD_DIR)
    return True

# Thực thi
if download_audio_files():
    print("✅ [Cell 4] Tiến trình tải danh sách file audio đã hoàn tất thành công.")
else:
    print("❌ [Cell 4] Tiến trình tải file đã thất bại. Vui lòng kiểm tra lại cấu hình.")

In [ ]:
# [CELL 5] XỬ LÝ TRANSCRIBE BẰNG DEEPGRAM API (SDK v7+ | MULTI-KEY ROTATION)
import json
import os
import random
from deepgram import DeepgramClient

def format_timestamp_to_srt(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millisecs = int(round((seconds - int(seconds)) * 1000))
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millisecs:03d}"

def generate_srt_content(utterances):
    srt_text = ""
    for index, utt in enumerate(utterances, start=1):
        start_time = format_timestamp_to_srt(utt.start)
        end_time = format_timestamp_to_srt(utt.end)
        srt_text += f"{index}\n{start_time} --> {end_time}\n{utt.transcript}\n\n"
    return srt_text

def create_client_with_rotation(available_keys):
    """
    Tạo DeepgramClient bằng cách random 1 key từ danh sách còn khả dụng.
    Trả về (client, key_được_dùng) hoặc raise Exception nếu hết key.
    """
    if not available_keys:
        raise Exception("Tất cả các Deepgram API key đã bị lỗi hoặc hết quota.")
    
    key = random.choice(available_keys)
    client = DeepgramClient(api_key=key)
    return client, key

def transcribe_with_fallback(file_path, available_keys):
    """
    Thử transcribe 1 file. Nếu key hiện tại lỗi thì tự động thử key khác.
    Trả về (response, key_thành_công) hoặc raise Exception nếu hết key.
    """
    # Shuffle để mỗi lần gọi thứ tự thử khác nhau, tránh dồn vào 1 key
    keys_to_try = available_keys.copy()
    random.shuffle(keys_to_try)

    last_error = None
    for key in keys_to_try:
        try:
            print(f"     [KEY] Đang dùng key: ...{key[-4:]}")
            client = DeepgramClient(api_key=key)

            with open(file_path, "rb") as audio_file:
                buffer_data = audio_file.read()

            response = client.listen.v1.media.transcribe_file(
                request=buffer_data,
                model="nova-2",
                detect_language=True,
                smart_format=True,
                utterances=True
            )
            # Thành công → trả về luôn
            return response, key

        except Exception as e:
            error_msg = str(e).lower()
            # Phân loại lỗi: nếu là lỗi key (auth/quota) thì thử key tiếp theo
            if any(keyword in error_msg for keyword in ["401", "403", "invalid", "quota", "limit", "unauthorized", "forbidden"]):
                print(f"     [!] Key ...{key[-6:]} gặp lỗi ({str(e)}). Thử key tiếp theo...")
                last_error = e
                continue
            else:
                # Lỗi khác (network, file, ...) → raise ngay, không cần thử key khác
                raise e

    raise Exception(f"Tất cả key đều thất bại. Lỗi cuối: {last_error}")

def process_audio_transcription():
    wav_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.wav')]

    if not wav_files:
        print("[!] Thư mục trống. Không có file .wav nào để xử lý.")
        return

    print(f"[*] Tổng cộng tìm thấy {len(wav_files)} file. Bắt đầu tiến trình Transcribe...")
    print(f"[*] Số lượng API key khả dụng: {len(DEEPGRAM_API_KEYS)}")

    # Danh sách key còn hoạt động — sẽ loại dần nếu key bị lỗi vĩnh viễn
    active_keys = DEEPGRAM_API_KEYS.copy()

    for idx, filename in enumerate(wav_files, start=1):
        file_path = os.path.join(DOWNLOAD_DIR, filename)
        base_name = os.path.splitext(filename)[0]
        json_path = os.path.join(OUTPUT_DIR, f"{base_name}_fixed.json")
        srt_path = os.path.join(OUTPUT_DIR, f"{base_name}_fixed.srt")

        print(f"\n[{idx}/{len(wav_files)}] Đang xử lý file: {filename}")

        try:
            response, used_key = transcribe_with_fallback(file_path, active_keys)

            # Lưu JSON
            with open(json_path, "w", encoding="utf-8") as json_out:
                json.dump(response.model_dump(), json_out, indent=4, default=str)

            # Xuất SRT từ utterances
            utterances = response.results.utterances if response.results else None
            if utterances:
                srt_content = generate_srt_content(utterances)
                with open(srt_path, "w", encoding="utf-8") as srt_out:
                    srt_out.write(srt_content)
                print(f"  -> ✅ Đã lưu JSON và SRT thành công.")
            else:
                print(f"  [!] Cảnh báo: Không tìm thấy utterances trong response của {filename}.")

        except Exception as e:
            print(f"  [!] ❌ Lỗi không thể phục hồi khi xử lý {filename}: {str(e)}")

process_audio_transcription()
print("\n✅ [Cell 5] Hoàn tất toàn bộ.")

In [ ]:
import os
import subprocess
import shutil

print("--- BẮT ĐẦU MODULE UPLOAD BẰNG RCLONE ---")

# Khai báo các đường dẫn
LOCAL_OUTPUT_DIR = OUTPUT_DIR
REMOTE_DRIVE_DIR = "codevcn-dell-pc:ae-b6/tiktok-beta-JAPAN/kaggle/movie-bridged-data/" 

# ĐƯỜNG DẪN CŨ (Chỉ Đọc - Read-only)
ORIGINAL_CONF = DATASET_RCLONE_CONF
# ĐƯỜNG DẪN MỚI (Cho phép Ghi đè - Writable)
WRITABLE_CONF = WORKING_RCLONE_CONF

# ==========================================
# KIỂM TRA: Có file nào trong thư mục output không?
# ==========================================
output_files = [
    f for f in os.listdir(LOCAL_OUTPUT_DIR)
    if os.path.isfile(os.path.join(LOCAL_OUTPUT_DIR, f))
] if os.path.isdir(LOCAL_OUTPUT_DIR) else []

if not output_files:
    print(f"⚠️ Không tìm thấy file nào trong thư mục output: {LOCAL_OUTPUT_DIR}")
    print("🚫 Bỏ qua bước upload lên Google Drive.")
else:
    # Sao chép tệp cấu hình ra ngoài thư mục working để rclone có thể cập nhật Token
    try:
        shutil.copyfile(ORIGINAL_CONF, WRITABLE_CONF)
    except Exception as e:
        print(f"Bỏ qua bước sao chép nếu tệp đã tồn tại. Chi tiết: {e}")
    
    print(f"\nĐang đồng bộ dữ liệu từ {LOCAL_OUTPUT_DIR} lên {REMOTE_DRIVE_DIR} ...")
    
    # Thực thi lệnh rclone sync (Trỏ cấu hình vào tệp Writable mới)
    # Lệnh sync sẽ tải tệp lên và xóa các tệp cũ trên đích không có trong thư mục nguồn
    command = f"rclone sync {LOCAL_OUTPUT_DIR} {REMOTE_DRIVE_DIR} --config {WRITABLE_CONF} -v"
    
    try:
        # Chạy lệnh thông qua hệ thống và dừng lại nếu có lỗi (check=True)
        subprocess.run(command, shell=True, check=True)
        
        print("\n✓ XUẤT SẮC! Toàn bộ tệp trong thư mục output đã được đẩy lên Google Drive.")
        print("✓ Các tệp cũ trên Google Drive đã được dọn dẹp thành công để nhường chỗ cho dữ liệu mới.")
        
        # Lấy đường dẫn chia sẻ của thư mục đích
        print("\nĐang tạo đường dẫn chia sẻ thư mục trên Google Drive...")
        link_command = f"rclone link {REMOTE_DRIVE_DIR} --config {WRITABLE_CONF}"
        
        # Chạy lệnh rclone link và bắt đầu ra (stdout)
        result = subprocess.run(link_command, shell=True, capture_output=True, text=True, check=True)
        folder_link = result.stdout.strip()
        
        print("==================================================")
        print(f"🔗 ĐƯỜNG DẪN THƯ MỤC CỦA BẠN: \n{folder_link}")
        print("==================================================")
        
    except subprocess.CalledProcessError as e:
        print(f"(!) Quá trình đồng bộ hoặc lấy đường dẫn đã thất bại. Vui lòng kiểm tra lại. Chi tiết lỗi: {e.stderr if e.stderr else e}")

In [ ]:
print("\nChuẩn bị kích hoạt Notebook tiếp theo trong chuỗi...")
send_webhook_to_server(
    index_type="start", # Sửa thành "mid" nếu là notebook ở giữa
    status="completed",
    next_notebook="hipbiquang/translate-srt-flow"
)

In [ ]:
# Gửi thông báo đến telegram

import requests

bot_token = '8943226169:AAFFQqX2YoZhKARtDPwyyNc83gZdmbnz260'
chat_id = '1331854393'

def send_telegram_message(message):
    """Hàm gửi tin nhắn văn bản đến Telegram"""
    if not bot_token or not chat_id:
        print("Thiếu Token hoặc Chat ID.")
        return
        
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {
        "chat_id": chat_id,
        "text": message,
        "parse_mode": "Markdown" # Cho phép định dạng chữ đậm, nghiêng
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            print("Đã gửi thông báo thành công đến Telegram!")
        else:
            print(f"Lỗi khi gửi thông báo: {response.text}")
    except Exception as e:
        print(f"Lỗi kết nối: {e}")

# Chạy thử nghiệm
send_telegram_message("✅ Đã transcribe file audio xong với Deepgram, chạy notebook tiếp theo: Dịch file SRT")